In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
import joblib
import urllib.request
import zipfile
import os

In [5]:
# Скачиваем архив
url = 'https://archive.ics.uci.edu/static/public/222/bank+marketing.zip'
zip_path = 'bank-marketing.zip'
urllib.request.urlretrieve(url, zip_path)

# Распаковываем архив
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall('bank-marketing')

# Указываем путь к CSV файлу
csv_path = os.path.join('bank-marketing', 'bank', 'bank-full.csv')

print(f"Файл сохранен по пути: {csv_path}")


Файл сохранен по пути: bank-marketing\bank\bank-full.csv


In [6]:
# Указываем полный путь к CSV файлу
csv_path = r'C:\Users\User\Типис\Untitled Folder\bank-marketing\bank-full.csv'

# Загружаем данные в DataFrame
df = pd.read_csv(csv_path, sep=';')

# Просмотр первых строк данных
print(df.head())

   age           job  marital  education default  balance housing loan  \
0   58    management  married   tertiary      no     2143     yes   no   
1   44    technician   single  secondary      no       29     yes   no   
2   33  entrepreneur  married  secondary      no        2     yes  yes   
3   47   blue-collar  married    unknown      no     1506     yes   no   
4   33       unknown   single    unknown      no        1      no   no   

   contact  day month  duration  campaign  pdays  previous poutcome   y  
0  unknown    5   may       261         1     -1         0  unknown  no  
1  unknown    5   may       151         1     -1         0  unknown  no  
2  unknown    5   may        76         1     -1         0  unknown  no  
3  unknown    5   may        92         1     -1         0  unknown  no  
4  unknown    5   may       198         1     -1         0  unknown  no  


In [ ]:
# Выбираем нужные столбцы
columns = ['age', 'job', 'marital', 'education', 'balance', 'housing', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'y']
df = df[columns]

# Разделение данных на обучающую, валидационную и тестовую выборки
df_train, df_temp = train_test_split(df, test_size=0.4, random_state=1)
df_val, df_test = train_test_split(df_temp, test_size=0.5, random_state=1)

print(f"Обучающая выборка: {df_train.shape}")
print(f"Валидационная выборка: {df_val.shape}")
print(f"Тестовая выборка: {df_test.shape}")

In [ ]:
numeric_columns = ['balance', 'day', 'duration', 'previous']
auc_scores = {}

# Вычисление AUC для каждой переменной
for col in numeric_columns:
    auc = roc_auc_score(df_train['y'], df_train[col])
    if auc < 0.5:
        auc = roc_auc_score(df_train['y'], -df_train[col])
    auc_scores[col] = auc

print("AUC для каждого признака:", auc_scores)

In [ ]:
# Применяем one-hot-encoding
train_dict = df_train.to_dict(orient='records')
val_dict = df_val.to_dict(orient='records')

dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(train_dict)
X_val = dv.transform(val_dict)

y_train = df_train['y'].values
y_val = df_val['y'].values

# Обучение логистической регрессии
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000)
model.fit(X_train, y_train)

# AUC на валидационной выборке
y_pred = model.predict_proba(X_val)[:, 1]
auc = roc_auc_score(y_val, y_pred)
print(f"AUC на валидационной выборке: {auc:.3f}")